In [1]:
import time
import rclpy
from rclpy.node import Node
from rclpy.action import ActionServer, GoalResponse, CancelResponse
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.executors import MultiThreadedExecutor
from turtlesim.action import RotateAbsolute

In [2]:
class TurtleRotateServer(Node):
    
    def __init__(self):
        super().__init__('turtle_rotate_server')
        self.server = ActionServer(
            self,
            RotateAbsolute,
            '/turtle1/rotate_absolute',
            execute_callback=self.execute_callback,
            goal_callback=self.goal_callback,
            cancel_callback=self.cancel_callback,
            callback_group=ReentrantCallbackGroup()

        )
        self.get_logger().info(f'Action server started.')
    
    def goal_callback(self, request):
        self.get_logger().info(f'Received goal request: {request}')
        return GoalResponse.ACCEPT
    
    def cancel_callback(self, request):
        self.get_logger().info(f'Received cancel request: {request}')
        return CancelResponse.ACCEPT
    
    async def execute_callback(self, goal_handle):
        feedback = RotateAbsolute.Feedback()
        feedback.remaining = 10.0

        for _ in range(10):
            if goal_handle.is_cancel_requested:
                goal_handle.canceled()
                self.get_logger().info('Action canceled.')
                return RotateAbsolute.Result()
        
            feedback.remaining -= 1
            goal_handle.publish_feedback(feedback)
            time.sleep(1)

        goal_handle.succeed()
        self.get_logger().info('Action succeed.')

        result = RotateAbsolute.Result()
        result.delta = 0.0
        return result

In [3]:
rclpy.init()
server = TurtleRotateServer()

[INFO] [1773901516.644234939] [turtle_rotate_server]: Action server started.


In [ ]:
executor = MultiThreadedExecutor()
# rclpy.spin_once(server, executor=executor)
rclpy.spin(server, executor=executor)

[INFO] [1773901538.194446542] [turtle_rotate_server]: Received goal request: turtlesim.action.RotateAbsolute_Goal(theta=3.140000104904175)
[INFO] [1773901548.211937777] [turtle_rotate_server]: Action succeed.
[INFO] [1773901573.455867216] [turtle_rotate_server]: Received goal request: turtlesim.action.RotateAbsolute_Goal(theta=0.0)
[INFO] [1773901583.481934352] [turtle_rotate_server]: Action succeed.
[INFO] [1773901677.427324040] [turtle_rotate_server]: Received goal request: turtlesim.action.RotateAbsolute_Goal(theta=3.140000104904175)
[INFO] [1773901682.993609877] [turtle_rotate_server]: Received goal request: turtlesim.action.RotateAbsolute_Goal(theta=0.0)
[INFO] [1773901687.448178832] [turtle_rotate_server]: Action succeed.
[INFO] [1773901693.014114425] [turtle_rotate_server]: Action succeed.


In [ ]:
executor.shutdown()

In [1]:
server.destroy_node()

NameError: name 'server' is not defined

In [ ]:
rclpy.shutdown()